## Nädala 7: UrbanStyle kliendibaasi RFM-segmenteerimine

**Meeskond:** Tooteanalüüsi osakond  
**Kuupäev:** 15.05.2026

---

# Kokkuvõte grupitööst

- Meie meeskond viis läbi UrbanStyle kliendiandmete RFM analüüsi Python pandas abil.
- Laadisime ja puhastasime müügi- ning kliendiandmed.
- Arvutasime Recency, Frequency ja Monetary mõõdikud.
- Lõime kaalutud kliendisegmenteerimise mudeli, kus Monetary väärtus sai 2x kaalu.
- Analüüsi eesmärk oli leida kõige väärtuslikumad kliendid ning tuvastada segmendid, mis vajavad kiiret tähelepanu.  



In [8]:
import pandas as pd
import plotly.express as px

### Roll A: Data Loading — andmete laadimine ja liitmine

In [9]:
# 1. Samm: 1. Loo uus Jupyter Notebook lahter. Impordi pandas ja supabase


"""
# Kood, kui impordiksin andmed Supabase'ist   
from supabase import create_client                                                         ## impordin create_client funktsiooni, et saaksime luua ühenduse Supabase'iga
import os
from dotenv import load_dotenv                                                                ## impordin load_dotenv funktsiooni, et saaksime laadida keskkonnamuutujaid .env failist  
load_dotenv() 

# 2. Samm: Loo ühendus Supabase'iga
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))                ## Loon ühenduse Supabase'iga, kasutades keskkonnamuutujaid SUPABASE_URL ja SUPABASE_KEY

# 3. Samm: sales tabeli laadimine
response_sales = supabase.table('sales').select('*').execute()                                ## Teeb andmebaasipäringu, et valida kõik veerud ja read müügitabelist
df_sales = pd.DataFrame(response_sales.data)                                                  ## Loob Pandas DataFrame'i, kasutades päringu tulemusi  

print(df_sales.shape)                                                                         ## Kuvab müügitabeli ridade ja veergude arvu
print(df_sales.head())                                                                        ## Kuvab müügitabeli esimesed read

# 4. Samm: customers andmete laadimine
response_customers = supabase.table('customers').select('*').execute()                        ## Teeb andmebaasipäringu, et valida kõik veerud ja read klientide tabelist
df_customers = pd.DataFrame(response_customers.data)                                          ## Loob Pandas DataFrame'i, kasutades päringu tulemusi

print(df_customers.shape)                                                                     ## Kuvab klientide tabeli ridade ja veergude arvu
print(df_customers.head())                                                                    ## Kuvab klientide tabeli esimesed read
"""

# Samm 2: Jätan vahele, kuna ei lae andmeid Supabase'ist

# Samm 3: Laen sales tabeli CSV failist
df_sales = pd.read_csv('sales.csv')                                                           ## Laen müügitabeli CSV failist Pandas DataFrame'i  

print("--Sales tabeli ridade ja veergude arv:", df_sales.shape)                                                                         ## Kuvab müügitabeli ridade ja veergude arvu
print(df_sales.head())                                                                        ## Kuvab müügitabeli esimesed read

# Samm 5: Laen customers tabeli CSV failist
df_customers = pd.read_csv('customers.csv')                                                   ## Laen klientide tabeli CSV failist Pandas DataFrame'i

print("--Customers tabeli ridade ja veergude arv:", df_customers.shape)                                                                     ## Kuvab klientide tabeli ridade ja veergude arvu
print(df_customers.head())                                                                    ## Kuvab klientide tabeli esimesed read

# Samm 5: Tabelite ühendamine
df_merged = pd.merge(df_sales, df_customers, on='customer_id', how='left')                    ## Ühendame müügi- ja klienditabelid customer_id

# Samm 6: Kontrollime ühendatud tabeli struktuuri
print("--Ühendatud tabeli ridade ja veergude arv:", df_merged.shape)                                                                        ## Kuvab ühendatud tabeli ridade ja veergude arvu
print(df_merged.head())                                                                       ## kuvab ühendatud tabeli veerunimed
print(df_merged.dtypes)                                                                       ## kuvab ühendatud tabeli veerunimed ja tüübid
required_cols = ['customer_id', 'sale_date', 'total_price', 'email']
print("--Nõutud veerud olemas:", all(c in df_merged.columns for c in required_cols))            ## Kontrollib, kas kõik nõutud veerud on olemas

--Sales tabeli ridade ja veergude arv: (15234, 11)
   sale_id        invoice_id   sale_date  customer_id  product_id  quantity  \
0        1  INV-202301-00001  2023-01-10       2588.0        1274         2   
1        2  INV-202301-00002  2023-01-16       4338.0        1207         2   
2        3  INV-202301-00003  2023-01-05       4673.0        1264         1   
3        4  INV-202301-00004  2023-01-02       4677.0        1341         3   
4        5  INV-202301-00005  2023-01-13       2390.0        1284         1   

   unit_price  total_price channel store_location payment_method  
0      234.79       469.58    pood        Tallinn          kaart  
1      241.13       482.26    pood          Pärnu      järelmaks  
2      258.46       221.19    pood          Pärnu      järelmaks  
3       45.21       135.63    pood          Tartu       sularaha  
4       99.57        99.57    pood          Tartu          kaart  
--Customers tabeli ridade ja veergude arv: (3150, 9)
   customer_id firs

### ROLL B: Data Cleaning - andmete puhastamine ja valideerimine

In [10]:
# Roll B: Data Cleaning — puhastamine ja valideerimine

print("Algne ridade arv:", df_merged.shape)
print("Algne unikaalsete klientide arv:", df_merged['customer_id'].nunique())

# 1. Duplikaatide kontroll ja eemaldamine
print("\nDuplikaatide arv:", df_merged.duplicated().sum())

df_cleaned = df_merged.drop_duplicates()

print("Pärast duplikaatide eemaldamist ridu:", df_cleaned.shape)
print("Pärast duplikaatide eemaldamist kliente:", df_cleaned['customer_id'].nunique())

# 2. NULL väärtuste kontroll
print("\nNULL väärtused veergudes:")
print(df_cleaned.isnull().sum())

print("\nKliente enne kriitiliste NULL-ide eemaldamist:", df_cleaned['customer_id'].nunique())

df_cleaned = df_cleaned.dropna(
    subset=['customer_id', 'sale_date', 'total_price']
)

print("Kliente pärast kriitiliste NULL-ide eemaldamist:", df_cleaned['customer_id'].nunique())

# 3. Kuupäevade muutmine õigesse formaati
df_cleaned['sale_date'] = pd.to_datetime(
    df_cleaned['sale_date'],
    errors='coerce'
)

print("\nKliente enne vigaste kuupäevade eemaldamist:", df_cleaned['customer_id'].nunique())

df_cleaned = df_cleaned.dropna(
    subset=['sale_date']
)

print("Kliente pärast vigaste kuupäevade eemaldamist:", df_cleaned['customer_id'].nunique())

# 4. Eemaldame müügid, mis on hilisemad kui viitekuupäev
print("\nKliente enne kuupäevafiltrit:", df_cleaned['customer_id'].nunique())

df_cleaned = df_cleaned[
    df_cleaned['sale_date'] <= pd.to_datetime('2025-02-28')
]

print("Kliente pärast kuupäevafiltrit:", df_cleaned['customer_id'].nunique())

# 5. Vigaste hindade eemaldamine
print("\nKliente enne hinnafiltrit:", df_cleaned['customer_id'].nunique())

df_cleaned = df_cleaned[
    df_cleaned['total_price'] > 0
]

print("Kliente pärast hinnafiltrit:", df_cleaned['customer_id'].nunique())

# 6. Puhastusraport
print("\n--- PUHASTUSRAPORT ---")
print("Lõplik ridade arv:", df_cleaned.shape)
print("Unikaalseid kliente:", df_cleaned['customer_id'].nunique())
print("Periood:", df_cleaned['sale_date'].min(), "kuni", df_cleaned['sale_date'].max())

print("\nValmis andmed analüüsiks:")
print(df_cleaned.head())

Algne ridade arv: (15234, 19)
Algne unikaalsete klientide arv: 2558

Duplikaatide arv: 4086
Pärast duplikaatide eemaldamist ridu: (11148, 19)
Pärast duplikaatide eemaldamist kliente: 2558

NULL väärtused veergudes:
sale_id                 0
invoice_id              0
sale_date               0
customer_id          1236
product_id              0
quantity                0
unit_price              0
total_price             0
channel                 0
store_location       3811
payment_method          0
first_name           1236
last_name            1236
email                2261
phone                1236
city                 1236
registration_date    1236
loyalty_tier         5205
birth_year           1236
dtype: int64

Kliente enne kriitiliste NULL-ide eemaldamist: 2558
Kliente pärast kriitiliste NULL-ide eemaldamist: 2558

Kliente enne vigaste kuupäevade eemaldamist: 2558
Kliente pärast vigaste kuupäevade eemaldamist: 2547

Kliente enne kuupäevafiltrit: 2547
Kliente pärast kuupäevafiltrit: 

### Roll C: RFM Analysis — Recency, Frequency, Monetary + segmendid

In [11]:
# Roll C: RFM Analysis — Recency, Frequency, Monetary + segmendid

import pandas as pd

# Viitekuupäev on fikseeritud, et tulemused oleksid võrreldavad
today = pd.to_datetime('2025-02-28')

# Samm 1: Arvuta Recency — viimane ostu kuupäev ja päevade arv
recency_df = df_cleaned.groupby('customer_id')['sale_date'].max().reset_index()

recency_df.columns = [
    'customer_id',
    'last_purchase'
]

recency_df['recency_days'] = (
    today - recency_df['last_purchase']
).dt.days

# Samm 2: Arvuta Frequency — ostude arv
frequency_df = (
    df_cleaned.groupby('customer_id')['sale_id']
    .count()
    .reset_index()
)

frequency_df.columns = [
    'customer_id',
    'frequency'
]

# Samm 3: Arvuta Monetary — kogukulutus
monetary_df = (
    df_cleaned.groupby('customer_id')['total_price']
    .sum()
    .reset_index()
)

monetary_df.columns = [
    'customer_id',
    'monetary_value'
]

# Samm 4: Liida R, F, M ühte tabelisse
rfm = pd.merge(
    recency_df[['customer_id', 'recency_days']],
    frequency_df,
    on='customer_id'
)

rfm = pd.merge(
    rfm,
    monetary_df,
    on='customer_id'
)

# Samm 5: Määra RFM skoorid (1–5)

# Recency puhul väiksem väärtus = parem klient
rfm['R_score'] = pd.qcut(
    rfm['recency_days'],
    5,
    labels=[5, 4, 3, 2, 1],
    duplicates='drop'
)

rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm['M_score'] = pd.qcut(
    rfm['monetary_value'],
    5,
    labels=[1, 2, 3, 4, 5]
)

# Samm 6: Klassikaline RFM skoor
rfm['RFM_Score'] = (
    rfm['R_score'].astype(int) +
    rfm['F_score'].astype(int) +
    rfm['M_score'].astype(int)
)

# Baastaseme segmenteerimine
def määra_segment(score):

    if score >= 13:
        return 'VIP Champions'

    elif score >= 10:
        return 'Loyal'

    elif score >= 7:
        return 'Potential'

    elif score >= 4:
        return 'At Risk'

    else:
        return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(
    määra_segment
)

# ---------------------------------------------------
# EDASIJÕUDNUD TASE — KAALUTUD RFM
# Monetary saab 2x kaalu
# ---------------------------------------------------

rfm['Weighted_RFM_Score'] = (
    rfm['R_score'].astype(int) +
    rfm['F_score'].astype(int) +
    (rfm['M_score'].astype(int) * 2)
)

# Täpsem segmenteerimine
def määra_advanced_segment(score):

    if score >= 17:
        return 'VIP Champions'

    elif score >= 14:
        return 'Loyal Customers'

    elif score >= 11:
        return 'Regular Customers'

    elif score >= 8:
        return 'New Customers'

    elif score >= 5:
        return 'At Risk'

    else:
        return 'Lost'

rfm['Advanced_Segment'] = rfm[
    'Weighted_RFM_Score'
].apply(
    määra_advanced_segment
)

# ---------------------------------------------------
# VÄLJUND: SEGMENTIDE KOKKUVÕTE
# ---------------------------------------------------

print("\n--- KAALUTUD RFM SEGMENTIDE KOKKUVÕTE ---")

advanced_summary = (
    rfm['Advanced_Segment']
    .value_counts(dropna=False)
    .reset_index()
)

advanced_summary.columns = [
    'Segment',
    'Klientide arv'
]

advanced_summary['Osakaal (%)'] = (
    advanced_summary['Klientide arv'] /
    advanced_summary['Klientide arv'].sum() * 100
).round(1)

print(advanced_summary)

# Kontroll: kas kõik kliendid said segmendi?
print(
    "\nSegmendita kliente:",
    rfm['Advanced_Segment'].isnull().sum()
)

# Näidisandmed
print("\nNäidis kliendiandmetest koos skooridega:")

print(
    rfm[
        [
            'customer_id',
            'recency_days',
            'frequency',
            'monetary_value',
            'Weighted_RFM_Score',
            'Advanced_Segment'
        ]
    ].head()
)

# ---------------------------------------------------
# EKSPORT CSV FAILINA
# ---------------------------------------------------

rfm.to_csv(
    'rfm_segments.csv',
    index=False
)

print(
    "\nCSV fail edukalt salvestatud: rfm_segments.csv"
)


--- KAALUTUD RFM SEGMENTIDE KOKKUVÕTE ---
             Segment  Klientide arv  Osakaal (%)
0  Regular Customers            543         21.4
1      VIP Champions            519         20.4
2      New Customers            498         19.6
3    Loyal Customers            480         18.9
4            At Risk            381         15.0
5               Lost            118          4.6

Segmendita kliente: 0

Näidis kliendiandmetest koos skooridega:
   customer_id  recency_days  frequency  monetary_value  Weighted_RFM_Score  \
0       2001.0            91          2          203.92                   7   
1       2004.0            71          2         1198.56                  13   
2       2005.0           148          4          959.60                  13   
3       2006.0           477          1          327.06                   4   
4       2007.0            29          1          318.63                   8   

    Advanced_Segment  
0            At Risk  
1  Regular Customers  
2  Re

### Roll D: Visualization — 3 diagrammi + äritõlgendus

In [12]:
# Loo diagramm 1 — tulpdiagramm: segmentide klientide arv.
# Kasuta px.bar() ja lisa pealkiri eesti keeles

# Kasutame uut või vana segmendi veergu
segment_col = 'Advanced_Segment' if 'Advanced_Segment' in rfm.columns else 'Segment'

# 1. Valmistame andmed ette
segment_counts = rfm[segment_col].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Klientide arv']

# 2. Sorteerime suuruse järgi
segment_counts = segment_counts.sort_values(
    'Klientide arv',
    ascending=False
)

# Segmentide järjekord
segment_order = segment_counts['Segment'].tolist()

# Arvutame koguarvu ja osakaalud
total_customers = segment_counts['Klientide arv'].sum()

segment_counts['Label'] = segment_counts.apply(
    lambda x: f"{int(x['Klientide arv'])} ({x['Klientide arv']/total_customers*100:.1f}%)",
    axis=1
)

# 3. Tulpdiagramm
fig1 = px.bar(
    segment_counts,
    x='Segment',
    y='Klientide arv',
    text='Label',

    title=f'UrbanStyle: Kliendisegmentide jaotus (kokku {total_customers} klienti)',

    labels={
        'Segment': 'Kliendisegment',
        'Klientide arv': 'Klientide arv'
    },

    color='Segment',

    category_orders={'Segment': segment_order},

    color_discrete_map={
   'VIP Champions': '#009B8D',      # UrbanStyle Teal
    'Loyal Customers': '#1A1A2E',    # UrbanStyle Navy
    'Regular Customers': '#32B4A9',  # Keskmine Teal
    'New Customers': '#A5DED7',      # Hele Teal
    'At Risk': '#FF9F43',            # Oranž (Hoiatus)
    'Lost': '#FF6B6B' 
    }
)

# 4. Visuaalne viimistlus
fig1.update_traces(
    textposition='outside',
    textfont_size=14,
    cliponaxis=False
)

fig1.update_layout(
    plot_bgcolor='white',
    font_family='Calibri',
    showlegend=False,

    xaxis_title='Kliendisegment',
    yaxis_title='Klientide arv',

    title_font_size=22
)

fig1.show()

In [13]:
# Loo diagramm 2 — hajuvusdiagramm:
# recency_days vs monetary_value, värvitud segmendi järgi, suurus = frequency

# 1. ETTEVALMISTUS
rfm_plot = rfm.copy()

if 'customer_id' not in rfm_plot.columns:
    rfm_plot = rfm_plot.reset_index()

segment_col = 'Advanced_Segment' if 'Advanced_Segment' in rfm_plot.columns else 'Segment'
recency_col = 'recency_days' if 'recency_days' in rfm_plot.columns else 'recency'
monetary_col = 'monetary_value' if 'monetary_value' in rfm_plot.columns else 'monetary'

segment_order = rfm_plot[segment_col].value_counts().index.tolist()
avg_monetary = rfm_plot[monetary_col].mean()

# Lisame kliendi nimed
customer_names = (
    df_cleaned[['customer_id', 'first_name', 'last_name']]
    .drop_duplicates()
)

rfm_plot = rfm_plot.merge(
    customer_names,
    on='customer_id',
    how='left'
)

rfm_plot['customer_name'] = (
    rfm_plot['first_name'].fillna('') + ' ' + rfm_plot['last_name'].fillna('')
).str.strip()

# 2. HAJUVUSDIAGRAMM
fig2 = px.scatter(
    rfm_plot,
    x=recency_col,
    y=monetary_col,
    color=segment_col,
    size='frequency',
    hover_name='customer_name',
    title='UrbanStyle kliendianalüüs: VIP-kliendid toovad suurima tulu ja on aktiivseimad',
    labels={
        recency_col: 'Päevi viimasest ostust',
        monetary_col: 'Kogukulutus (EUR)',
        segment_col: 'Kliendisegment',
        'frequency': 'Ostude arv'
    },
    category_orders={segment_col: segment_order},
    color_discrete_map={
     'VIP Champions': '#009B8D',      # UrbanStyle Teal
    'Loyal Customers': '#1A1A2E',    # UrbanStyle Navy
    'Regular Customers': '#32B4A9',  # Keskmine Teal
    'New Customers': '#A5DED7',      # Hele Teal
    'At Risk': '#FF9F43',            # Oranž (Hoiatus)
    'Lost': '#FF6B6B' 
    },
    hover_data={
        'customer_id': False,
        'first_name': False,
        'last_name': False
    },
    size_max=40
)

# 3. VISUAALNE VIIMISTLUS
fig2.update_layout(
    plot_bgcolor='white',
    font_family='Calibri',

    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor='#E5E7EB',
        linewidth=1,
        range=[-50, rfm_plot[recency_col].max() * 1.1]
    ),

    yaxis=dict(
        showgrid=True,
        gridcolor='#F3F4F6',
        griddash='dot',
        showline=True,
        linecolor='#E5E7EB',
        linewidth=1,
        tickvals=[0, 5000, 10000, 15000, 20000, 25000, 30000],
        ticktext=['0', '5k', '10k', '15k', '20k', '25k', '30k'],
        range=[-1000, 32000]
    ),

    margin=dict(t=140, l=70, r=40, b=60)
)

# 4. KESKMISE KULUTUSE JOON
fig2.add_hline(
    y=avg_monetary,
    line_dash="dot",
    line_color="#E5E7EB",
    annotation_text=f"Keskmine: €{avg_monetary:.0f}",
    annotation_position="top right",
    annotation_font_color="#6B7280"
)

# 5. VIP ANNOTATSIOON
vip_top_client = rfm_plot[
    rfm_plot[segment_col] == 'VIP Champions'
].nlargest(1, monetary_col)

if not vip_top_client.empty:
    target_x = vip_top_client[recency_col].iloc[0]
    target_y = vip_top_client[monetary_col].iloc[0]

    fig2.add_annotation(
        x=target_x + 6,
        y=target_y + 2800,
        text="VIP Champions: meie väärtuslikem segment",
        showarrow=True,
        arrowhead=0,
        arrowcolor="#009B8D",
        arrowwidth=1.5,
        ax=70,
        ay=-40,
        font=dict(color="#009B8D", size=13),
        bgcolor="rgba(255, 255, 255, 0.7)",
        bordercolor="rgba(0,0,0,0)",
        xanchor='center',
        yanchor='bottom'
    )

fig2.show()

In [14]:
# Loo diagramm 3 — top 10 VIP klienti kogukulutuse järgi.
# Kasuta rfm[rfm['Segment'] == 'VIP Champions'].nlargest(10, 'monetary_value')

# 1. ETTEVALMISTUS
rfm_vip = rfm.copy()

if 'customer_id' not in rfm_vip.columns:
    rfm_vip = rfm_vip.reset_index()

segment_col = 'Advanced_Segment' if 'Advanced_Segment' in rfm_vip.columns else 'Segment'
monetary_col = 'monetary_value' if 'monetary_value' in rfm_vip.columns else 'monetary'

# 2. TOP 10 VIP klienti
top_vip = (
    rfm_vip[rfm_vip[segment_col] == 'VIP Champions']
    .nlargest(10, monetary_col)
)

# 3. Lisame kliendi nimed
customer_names = (
    df_cleaned[['customer_id', 'first_name', 'last_name']]
    .drop_duplicates()
)

top_vip = top_vip.merge(
    customer_names,
    on='customer_id',
    how='left'
)

top_vip['customer_name'] = (
    top_vip['first_name'].fillna('') + ' ' + top_vip['last_name'].fillna('')
).str.strip()

top_vip['customer_name'] = top_vip['customer_name'].replace('', pd.NA)

top_vip['customer_name'] = top_vip['customer_name'].fillna(
    top_vip['customer_id'].astype(str)
)

# 4. Loome diagrammi
fig3 = px.bar(
    top_vip,
    x=monetary_col,
    y='customer_name',
    orientation='h',
    title='UrbanStyle: TOP 10 VIP-klienti kogukulutuse järgi',
    labels={
        monetary_col: 'Kogukulutus (EUR)',
        'customer_name': ''
    },
    text=monetary_col,
    color_discrete_sequence=['#009B8D']
)

# 5. Tulpade otsas täpne summa
fig3.update_traces(
    texttemplate='€%{x:,.0f}',
    textposition='outside',
    cliponaxis=False
)

# 6. Visuaalne viimistlus
fig3.update_layout(
    plot_bgcolor='white',
    font_family='Calibri',
    showlegend=False,
    bargap=0.25,

    xaxis=dict(
        showgrid=False,
        showticklabels=False,
        title=''
    ),

    yaxis=dict(
        title='',
        autorange='reversed',
        ticklabelstandoff=15
    ),

    margin=dict(t=90, l=160, r=120, b=60)
)

fig3.show()

## Peamised leiud ja äritõlgendus

- **519 VIP Champions klienti (20.4%)** moodustavad UrbanStyle’i kõige väärtuslikuma kliendisegmendi ning nende kogukulutus on oluliselt kõrgem kui teistel klientidel.
- TOP 10 VIP kliendi kogukulutus jääb vahemikku umbes **20 000–28 000 eurot**, mis näitab tugevat käibe koondumist väikese kliendigrupi kätte.
- **381 At Risk klienti (15.0%)** vajavad kiiret tähelepanu, sest nende ostuaktiivsus on langenud ning osa väärtuslikest klientidest võib olla churn-riskis.
- Segmentide jaotus näitab, et erinevad kliendigrupid vajavad erinevaid turundus- ja kommunikatsioonistrateegiaid.
---

# Äritõlgendus Markole

Analüüs näitab, et UrbanStyle’i kliendibaas koosneb väga erineva väärtuse ja ostukäitumisega segmentidest, mistõttu tuleks turundus- ja lojaalsustegevused kohandada vastavalt kliendisegmendile. Kõige suurema ärilise mõju annavad väärtuslike klientide hoidmine ning langeva aktiivsusega klientide tagasivõitmine.

# Soovitused Markole

## 1. Luua VIP programm

Pakkuda VIP Champions klientidele:
- early access kampaaniaid,
- personaalseid soodustusi,
- lojaalsusprogramme,
- eksklusiivseid pakkumisi.

Eesmärk on hoida kõige väärtuslikumad kliendid aktiivsena.

---

## 2. Käivitada win-back kampaania

Suunata At Risk klientidele:
- personaalsed “Me igatseme sind” pakkumised,
- ajaliselt piiratud soodustused,
- personaliseeritud e-mailid varasema ostukäitumise põhjal.

Eesmärk on taastada nende ostuaktiivsus ja vähendada churn-riski.

---

## 3. Arendada New Customers onboardingut

Luua uutele klientidele:
- welcome e-mailide seeria,
- esmaostu soodustused,
- cross-sell soovitused.

Eesmärk on suurendada korduvoste ja kasvatada lojaalsust.

---

# Mis meid üllatas?

Kõrge väärtusega At Risk kliente oli rohkem, kui algselt eeldasime — kokku 381 klienti ehk 15% kogu kliendibaasist. See viitab sellele, et oluline osa käibest võib olla churn-riskis ning vajab kiiret tähelepanu.